In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
# Imports & load data
import pandas as pd
import numpy as np

# point to your filtered file
file_path = '/content/drive/MyDrive/MRP/news_missing_summary_filled_2016_2023.csv'

# read, parsing the Date column as datetime
df = pd.read_csv(file_path, parse_dates=['Date'])

In [ ]:
df.head()

,Date,Stock_symbol,Textrank_summary
0,2016-01-06 00:00:00+00:00,A,Deutsche Bank Initiates Coverage on Agilent Te...
1,2016-01-06 00:00:00+00:00,A,Agilent to pay Enzo Bio $9M to settle patent f...
2,2016-01-07 00:00:00+00:00,A,Benzinga's Top Upgrades
3,2016-01-07 00:00:00+00:00,A,Cowen &amp; Company Upgrades Agilent Technolog...
4,2016-02-05 00:00:00+00:00,A,Tracking William Von Mueffling's Cantillon Cap...


In [ ]:
!pip install transformers tqdm --quiet

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from tqdm import tqdm

In [ ]:
# File path
file_path = '/content/drive/MyDrive/MRP/news_missing_summary_filled_2016_2023.csv'

# Read CSV
df = pd.read_csv(file_path, parse_dates=['Date'])
df.head()

,Date,Stock_symbol,Textrank_summary
0,2016-01-06 00:00:00+00:00,A,Deutsche Bank Initiates Coverage on Agilent Te...
1,2016-01-06 00:00:00+00:00,A,Agilent to pay Enzo Bio $9M to settle patent f...
2,2016-01-07 00:00:00+00:00,A,Benzinga's Top Upgrades
3,2016-01-07 00:00:00+00:00,A,Cowen &amp; Company Upgrades Agilent Technolog...
4,2016-02-05 00:00:00+00:00,A,Tracking William Von Mueffling's Cantillon Cap...


In [ ]:
model_name = "yiyanghkust/finbert-tone"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

finbert = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    truncation=True,
    max_length=512,
    padding=True,
    device=0
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
def get_sentiment(texts):
    results = finbert(texts)
    return [(r['label'], r['score']) for r in results]

In [ ]:
# Clean and prepare text
summaries = df['Textrank_summary'].fillna('').tolist()
sentiments = []
batch_size = 32

for i in tqdm(range(0, len(summaries), batch_size), desc="Running FinBERT"):
    batch = summaries[i:i+batch_size]
    try:
        batch_results = get_sentiment(batch)
        sentiments.extend(batch_results)
    except Exception as e:
        print(f"Error at batch {i}: {e}")
        sentiments.extend([('neutral', 0.0)] * len(batch))  # fallback

Running FinBERT:   0%|          | 0/125822 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

Running FinBERT: 100%|██████████| 125822/125822 [11:48:40<00:00,  2.96it/s]


In [ ]:
# Add FinBERT results to the dataframe
df[['sentiment_label', 'sentiment_score']] = pd.DataFrame(sentiments, index=df.index)

# Convert sentiment label to lowercase before mapping
df['sentiment_numeric'] = df['sentiment_label'].str.lower().map({
    'positive': 1,
    'neutral': 0,
    'negative': -1
})

# Flag for article presence
df['has_article'] = 1

# Check results
df.head()

NameError: name 'pd' is not defined

In [ ]:
output_path = '/content/drive/MyDrive/MRP/news_textrank_with_sentiment.csv'
df.to_csv(output_path, index=False)
print(f"Sentiment analysis complete. File saved to: {output_path}")

Sentiment analysis complete. File saved to: /content/drive/MyDrive/MRP/news_textrank_with_sentiment.csv
